# Data Exploration
This notebook provides a standard data exploration workflow (load, inspect, summarize, visualize, and save).

In [ ]:
from pathlib import Path
import os
import warnings

# Find repo root (look for build_foundation_lcms.py)
def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for parent in [cur] + list(cur.parents):
        if (parent / "build_foundation_lcms.py").exists():
            return parent
    return cur

BASE_PATH = find_repo_root(Path.cwd())
DATA_RAW = BASE_PATH / "data" / "raw"
DATA_MZML = BASE_PATH / "data" / "mzml"
DATA_VOXEL = BASE_PATH / "data" / "voxel"
DATA_WINDOWS = BASE_PATH / "data" / "windows"

print("BASE_PATH:", BASE_PATH)
print("RAW:", DATA_RAW)
print("MZML:", DATA_MZML)
print("VOXEL:", DATA_VOXEL)
print("WINDOWS:", DATA_WINDOWS)

try:
    import pandas as pd
except Exception as exc:
    pd = None
    warnings.warn(f"pandas not available: {exc}")

BASE_PATH: /home/simonp/FoundationMSMS
RAW: /home/simonp/FoundationMSMS/data/raw
MZML: /home/simonp/FoundationMSMS/data/mzml
VOXEL: /home/simonp/FoundationMSMS/data/voxel
WINDOWS: /home/simonp/FoundationMSMS/data/windows


## Project File Type Inventory
Summarize file extensions and acquisition keywords (e.g., DDA/DIA/ETD/HCD) per dataset.

In [ ]:
from collections import Counter, defaultdict

def scan_project_files(base_raw: Path, max_files: int = 2000):
    records = []
    if not base_raw.exists():
        print("RAW directory not found:", base_raw)
        return records
    for source_dir in sorted(base_raw.iterdir()):
        if not source_dir.is_dir():
            continue
        source = source_dir.name
        for dataset_dir in sorted(source_dir.iterdir()):
            if not dataset_dir.is_dir():
                continue
            dataset = dataset_dir.name
            files = []
            for p in dataset_dir.rglob("*"):
                if p.is_file():
                    files.append(p)
                    if len(files) >= max_files:
                        break
            for p in files:
                name = p.name
                ext = p.suffix.lower()
                tokens = []
                upper = name.upper()
                for key in ["DDA", "DIA", "HCD", "ETD", "CID", "MS1", "MS2"]:
                    if key in upper:
                        tokens.append(key)
                records.append({
                    "source": source,
                    "dataset": dataset,
                    "ext": ext if ext else "(no_ext)",
                    "tokens": ",".join(tokens) if tokens else "(none)",
                })
    return records

records = scan_project_files(DATA_RAW, max_files=5000)
if not records:
    print("No files found under RAW.")
else:
    if pd is not None:
        df_files = pd.DataFrame(records)
        display(df_files.groupby(["source", "dataset", "ext"]).size().reset_index(name="count").head(50))
        display(df_files.groupby(["source", "dataset", "tokens"]).size().reset_index(name="count").head(50))
    else:
        ext_counts = Counter((r["source"], r["dataset"], r["ext"]) for r in records)
        token_counts = Counter((r["source"], r["dataset"], r["tokens"]) for r in records)
        print("Top extension counts:")
        for k, v in ext_counts.items():
            print(k, v)
        print("Top token counts:")
        for k, v in token_counts.items():
            print(k, v)

,source,dataset,ext,count
0,massive,MSV000082648,(no_ext),1
1,massive,MSV000082648,.gz,168
2,massive,MSV000082648,.tsv,1
3,massive,MSV000082648,.txt,168
4,massive,MSV000082648,.xml,1
5,pride,PXD010595,.raw,13
6,pride,PXD012353,.raw,24
7,pride,PXD021874,.raw,62
8,pride,PXD028735,.raw,29
9,pride,PXD028735,.scan,76


,source,dataset,tokens,count
0,massive,MSV000082648,(none),339
1,pride,PXD010595,DDA,5
2,pride,PXD010595,ETD,3
3,pride,PXD010595,HCD,5
4,pride,PXD012353,(none),24
5,pride,PXD021874,(none),62
6,pride,PXD028735,(none),164
7,pride,PXD028735,DDA,73
8,pride,PXD028735,DIA,6


In [ ]:
# Build dataframes for mzML and voxel (npz) files
mzml_records = scan_project_files(DATA_MZML, max_files=5000)
voxel_records = scan_project_files(DATA_VOXEL, max_files=5000)

if pd is None:
    print("pandas is required for dataframe outputs.")
else:
    df_mzml = pd.DataFrame(mzml_records)
    df_voxel = pd.DataFrame(voxel_records)

    # Optional quick summaries
    display(df_mzml.groupby(["source", "dataset", "ext"]).size().reset_index(name="count").head(50))
    display(df_voxel.groupby(["source", "dataset", "ext"]).size().reset_index(name="count").head(50))

mzml_records = [r for r in mzml_records if r["ext"] == ".mzml"]
voxel_records = [r for r in voxel_records if r["ext"] == ".npz"]

,source,dataset,ext,count
0,pride,PXD010595,.mzml,12
1,pride,PXD012353,.mzml,23


,source,dataset,ext,count
0,pride,PXD010595,.npz,12
1,pride,PXD012353,.npz,23


In [ ]:
mzml_records

[{'source': 'pride', 'dataset': 'PXD010595', 'ext': '.mzml', 'tokens': 'DDA'},
 {'source': 'pride', 'dataset': 'PXD010595', 'ext': '.mzml', 'tokens': 'DDA'},
 {'source': 'pride', 'dataset': 'PXD010595', 'ext': '.mzml', 'tokens': 'DDA'},
 {'source': 'pride', 'dataset': 'PXD010595', 'ext': '.mzml', 'tokens': 'ETD'},
 {'source': 'pride', 'dataset': 'PXD010595', 'ext': '.mzml', 'tokens': 'ETD'},
 {'source': 'pride', 'dataset': 'PXD010595', 'ext': '.mzml', 'tokens': 'ETD'},
 {'source': 'pride', 'dataset': 'PXD010595', 'ext': '.mzml', 'tokens': 'DDA'},
 {'source': 'pride', 'dataset': 'PXD010595', 'ext': '.mzml', 'tokens': 'HCD'},
 {'source': 'pride', 'dataset': 'PXD010595', 'ext': '.mzml', 'tokens': 'HCD'},
 {'source': 'pride', 'dataset': 'PXD010595', 'ext': '.mzml', 'tokens': 'HCD'},
 {'source': 'pride', 'dataset': 'PXD010595', 'ext': '.mzml', 'tokens': 'DDA'},
 {'source': 'pride', 'dataset': 'PXD010595', 'ext': '.mzml', 'tokens': 'HCD'},
 {'source': 'pride',
  'dataset': 'PXD012353',
  'ex

In [ ]:
# Ensure voxel dataset table exists before Step 1
if pd is None:
    print("pandas is required for dataframe outputs.")
else:
    if "df_voxel_files" not in globals():
        rows = []
        for source_dir in sorted(DATA_VOXEL.iterdir()):
            if not source_dir.is_dir():
                continue
            source = source_dir.name
            for dataset_dir in sorted(source_dir.iterdir()):
                if not dataset_dir.is_dir():
                    continue
                dataset = dataset_dir.name
                for p in dataset_dir.glob("*.npz"):
                    rows.append({
                        "source": source,
                        "dataset": dataset,
                        "file": p.name,
                        "path": str(p),
                        "size_bytes": p.stat().st_size,
                    })
        df_voxel_files = pd.DataFrame(rows)
    display(df_voxel_files.head())

,source,dataset,file,path,size_bytes
0,pride,PXD010595,01974c_BF1-TUM_missing_first_6_01_01-3xHCD-1h-...,/home/simonp/FoundationMSMS/data/voxel/pride/P...,19396816
1,pride,PXD010595,02208a_GD12-TUM_second_addon_48_01_01-DDA-1h-R...,/home/simonp/FoundationMSMS/data/voxel/pride/P...,189699874
2,pride,PXD010595,02080d_GB6-TUM_isoform_114_01_01-ETD-1h-R1.npz,/home/simonp/FoundationMSMS/data/voxel/pride/P...,4891935
3,pride,PXD010595,02208a_GA4-TUM_second_addon_4_01_01-DDA-1h-R1.npz,/home/simonp/FoundationMSMS/data/voxel/pride/P...,106902954
4,pride,PXD010595,02208a_GB6-TUM_second_addon_18_01_01-3xHCD-1h-...,/home/simonp/FoundationMSMS/data/voxel/pride/P...,16224221


## 1. Create Dataset from Voxel Index
Use the voxel file table as the working dataframe.

In [ ]:
if pd is None:
    print("pandas is required for dataframe outputs.")
elif "df_voxel_files" not in globals():
    print("Run the Voxel Dataset Index cell first to create df_voxel_files.")
else:
    df = df_voxel_files.copy()
    display(df.head())

,source,dataset,file,path,size_bytes
0,pride,PXD010595,01974c_BF1-TUM_missing_first_6_01_01-3xHCD-1h-...,/home/simonp/FoundationMSMS/data/voxel/pride/P...,19396816
1,pride,PXD010595,02208a_GD12-TUM_second_addon_48_01_01-DDA-1h-R...,/home/simonp/FoundationMSMS/data/voxel/pride/P...,189699874
2,pride,PXD010595,02080d_GB6-TUM_isoform_114_01_01-ETD-1h-R1.npz,/home/simonp/FoundationMSMS/data/voxel/pride/P...,4891935
3,pride,PXD010595,02208a_GA4-TUM_second_addon_4_01_01-DDA-1h-R1.npz,/home/simonp/FoundationMSMS/data/voxel/pride/P...,106902954
4,pride,PXD010595,02208a_GB6-TUM_second_addon_18_01_01-3xHCD-1h-...,/home/simonp/FoundationMSMS/data/voxel/pride/P...,16224221


## 2. Inspect Schema, Types, and Missing Values
Review shape, columns, dtypes, and missing counts.

In [ ]:
if df is None:
    print("Load a dataset first.")
else:
    print("Shape:", df.shape)
    display(df.dtypes)
    missing = df.isna().sum().sort_values(ascending=False)
    display(missing.head(20))

Shape: (35, 5)


source          str
dataset         str
file            str
path            str
size_bytes    int64
dtype: object

source        0
dataset       0
file          0
path          0
size_bytes    0
dtype: int64

## 3. Summary Statistics for Numeric Columns
Compute summary stats plus skew and kurtosis.

In [ ]:
if df is None:
    print("Load a dataset first.")
else:
    num_df = df.select_dtypes(include="number")
    if num_df.empty:
        print("No numeric columns detected.")
    else:
        display(num_df.describe())
        display(num_df.skew(numeric_only=True).to_frame("skew"))
        display(num_df.kurtosis(numeric_only=True).to_frame("kurtosis"))

,size_bytes
count,3.500000e+01
mean,6.232775e+07
std,5.223205e+07
min,1.020541e+06
25%,2.151963e+07
50%,5.276182e+07
75%,7.789548e+07
max,1.937753e+08


,skew
size_bytes,1.179571


,kurtosis
size_bytes,1.082695


## 4. Frequency Tables for Categorical Columns
Summarize value counts for key categorical fields.

In [ ]:
if df is None:
    print("Load a dataset first.")
else:
    cat_df = df.select_dtypes(exclude="number")
    if cat_df.empty:
        print("No categorical columns detected.")
    else:
        for col in cat_df.columns[:5]:
            vc = cat_df[col].value_counts(dropna=False).head(10)
            display(vc.to_frame(col))

,source
source,
pride,35


,dataset
dataset,
PXD012353,23
PXD010595,12


,file
file,
01974c_BF1-TUM_missing_first_6_01_01-3xHCD-1h-R4.npz,1
02208a_GD12-TUM_second_addon_48_01_01-DDA-1h-R1.npz,1
02080d_GB6-TUM_isoform_114_01_01-ETD-1h-R1.npz,1
02208a_GA4-TUM_second_addon_4_01_01-DDA-1h-R1.npz,1
02208a_GB6-TUM_second_addon_18_01_01-3xHCD-1h-R1.npz,1
02080d_GA6-TUM_isoform_102_01_01-ETD-1h-R1.npz,1
02101a_GA1-TUM_proteo_TMT_1_01_01-2xIT_2xHCD-1h-R1.npz,1
02097a_BH6-TUM_isoform_90_01_01-DDA-1h-R3.npz,1
02097a_BA6-TUM_isoform_6_01_01-ETD-1h-R3.npz,1


,path
path,
/home/simonp/FoundationMSMS/data/voxel/pride/PXD010595/01974c_BF1-TUM_missing_first_6_01_01-3xHCD-1h-R4.npz,1
/home/simonp/FoundationMSMS/data/voxel/pride/PXD010595/02208a_GD12-TUM_second_addon_48_01_01-DDA-1h-R1.npz,1
/home/simonp/FoundationMSMS/data/voxel/pride/PXD010595/02080d_GB6-TUM_isoform_114_01_01-ETD-1h-R1.npz,1
/home/simonp/FoundationMSMS/data/voxel/pride/PXD010595/02208a_GA4-TUM_second_addon_4_01_01-DDA-1h-R1.npz,1
/home/simonp/FoundationMSMS/data/voxel/pride/PXD010595/02208a_GB6-TUM_second_addon_18_01_01-3xHCD-1h-R1.npz,1
/home/simonp/FoundationMSMS/data/voxel/pride/PXD010595/02080d_GA6-TUM_isoform_102_01_01-ETD-1h-R1.npz,1
/home/simonp/FoundationMSMS/data/voxel/pride/PXD010595/02101a_GA1-TUM_proteo_TMT_1_01_01-2xIT_2xHCD-1h-R1.npz,1
/home/simonp/FoundationMSMS/data/voxel/pride/PXD010595/02097a_BH6-TUM_isoform_90_01_01-DDA-1h-R3.npz,1
/home/simonp/FoundationMSMS/data/voxel/pride/PXD010595/02097a_BA6-TUM_isoform_6_01_01-ETD-1h-R3.npz,1


## 5. Basic Visualizations (Histograms, Boxplots, Pairplot)
Plot distributions and pairwise relationships.

In [ ]:
if df is None:
    print("Load a dataset first.")
else:
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        sns.set_theme(style="whitegrid")
        num_cols = df.select_dtypes(include="number").columns.tolist()
        if num_cols:
            df[num_cols].hist(figsize=(12, 8), bins=30)
            plt.tight_layout()
            plt.show()
            sns.boxplot(data=df[num_cols].melt(var_name="col", value_name="val"), x="col", y="val")
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
            if len(num_cols) >= 2:
                sns.pairplot(df[num_cols].sample(min(len(df), 500)))
                plt.show()
        else:
            print("No numeric columns to plot.")
    except Exception as exc:
        print(f"Plotting skipped: {exc}")

Plotting skipped: No module named 'matplotlib'


## 6. Correlation Analysis and Heatmap
Compute a correlation matrix and visualize it.

In [ ]:
if df is None:
    print("Load a dataset first.")
else:
    num_df = df.select_dtypes(include="number")
    if num_df.shape[1] < 2:
        print("Need at least two numeric columns for correlation.")
    else:
        corr = num_df.corr(numeric_only=True)
        display(corr.head())
        try:
            import matplotlib.pyplot as plt
            import seaborn as sns
            sns.heatmap(corr, cmap="viridis", center=0)
            plt.tight_layout()
            plt.show()
        except Exception as exc:
            print(f"Heatmap skipped: {exc}")

Need at least two numeric columns for correlation.


## 7. Outlier Detection with IQR/Z-Score
Identify potential outliers for numeric columns.

In [ ]:
import numpy as np

if df is None:
    print("Load a dataset first.")
else:
    num_df = df.select_dtypes(include="number")
    if num_df.empty:
        print("No numeric columns detected.")
    else:
        z_scores = (num_df - num_df.mean()) / num_df.std(ddof=0)
        z_outliers = (z_scores.abs() > 3).sum().sort_values(ascending=False)
        display(z_outliers.to_frame("zscore_outliers"))

        q1 = num_df.quantile(0.25)
        q3 = num_df.quantile(0.75)
        iqr = q3 - q1
        iqr_outliers = ((num_df < (q1 - 1.5 * iqr)) | (num_df > (q3 + 1.5 * iqr))).sum()
        display(iqr_outliers.to_frame("iqr_outliers"))

,zscore_outliers
size_bytes,0


,iqr_outliers
size_bytes,3


## 7b. Voxel Stats and Visualization
Summarize voxel files and visualize intensity distributions.

In [ ]:
SOURCE = "pride"  # or "massive"
DATASET_ID = "PXD012353"  # set dataset to explore
MAX_VOXEL_FILES = 20

voxel_dir = DATA_VOXEL / SOURCE / DATASET_ID
if not voxel_dir.exists():
    print("Voxel dir not found:", voxel_dir)
else:
    voxel_files = sorted(voxel_dir.glob("*.npz"))
    print("Voxel files:", len(voxel_files))
    sample_files = voxel_files[:MAX_VOXEL_FILES]
    stats = []
    for p in sample_files:
        try:
            npz = np.load(p)
            vals = npz["vals"]
            stats.append({
                "file": p.name,
                "count": int(vals.size),
                "min": float(vals.min()) if vals.size else 0.0,
                "max": float(vals.max()) if vals.size else 0.0,
                "mean": float(vals.mean()) if vals.size else 0.0,
                "p95": float(np.percentile(vals, 95)) if vals.size else 0.0,
            })
        except Exception as exc:
            print("Skip", p.name, exc)
    if stats:
        if pd is not None:
            display(pd.DataFrame(stats))
        else:
            for row in stats:
                print(row)
        all_vals = []
        for row in stats:
            p = voxel_dir / row["file"]
            try:
                vals = np.load(p)["vals"]
                if vals.size:
                    all_vals.append(vals)
            except Exception:
                continue
        if all_vals:
            concat = np.concatenate(all_vals)
            try:
                import matplotlib.pyplot as plt
                plt.hist(concat, bins=50)
                plt.title("Voxel intensity distribution")
                plt.tight_layout()
                plt.show()
            except Exception as exc:
                print("Plotting skipped:", exc)

Voxel files: 23


## Voxel Dataset Index
Build a dataset table from voxel .npz files for analysis.

In [ ]:
# Build a dataframe of all voxel files
MAX_VOXEL_FILES_PER_DATASET = None  # set an int to limit per dataset
rows = []
for source_dir in sorted(DATA_VOXEL.iterdir()):
    if not source_dir.is_dir():
        continue
    source = source_dir.name
    for dataset_dir in sorted(source_dir.iterdir()):
        if not dataset_dir.is_dir():
            continue
        dataset = dataset_dir.name
        count = 0
        for p in dataset_dir.glob("*.npz"):
            rows.append({
                "source": source,
                "dataset": dataset,
                "file": p.name,
                "path": str(p),
                "size_bytes": p.stat().st_size,
            })
            count += 1
            if MAX_VOXEL_FILES_PER_DATASET is not None and count >= MAX_VOXEL_FILES_PER_DATASET:
                break

if pd is None:
    print("pandas is required for dataframe outputs.")
else:
    df_voxel_files = pd.DataFrame(rows)
    display(df_voxel_files.head())
    df_voxel_index = df_voxel_files.groupby(["source", "dataset"]).agg(
        file_count=("file", "count"),
        total_bytes=("size_bytes", "sum"),
    ).reset_index()
    display(df_voxel_index.sort_values(["source", "dataset"]).head(50))

,source,dataset,file,path,size_bytes
0,pride,PXD010595,01974c_BF1-TUM_missing_first_6_01_01-3xHCD-1h-...,/home/simonp/FoundationMSMS/data/voxel/pride/P...,19396816
1,pride,PXD010595,02208a_GD12-TUM_second_addon_48_01_01-DDA-1h-R...,/home/simonp/FoundationMSMS/data/voxel/pride/P...,189699874
2,pride,PXD010595,02080d_GB6-TUM_isoform_114_01_01-ETD-1h-R1.npz,/home/simonp/FoundationMSMS/data/voxel/pride/P...,4891935
3,pride,PXD010595,02208a_GA4-TUM_second_addon_4_01_01-DDA-1h-R1.npz,/home/simonp/FoundationMSMS/data/voxel/pride/P...,106902954
4,pride,PXD010595,02208a_GB6-TUM_second_addon_18_01_01-3xHCD-1h-...,/home/simonp/FoundationMSMS/data/voxel/pride/P...,16224221


,source,dataset,file_count,total_bytes
0,pride,PXD010595,12,953244527
1,pride,PXD012353,23,1228226790


## Voxel Tensor Loader
Load voxel .npz files into sparse coords/vals (and optional dense tensors).

In [ ]:
# Load voxel files into sparse tensors (coords, vals).
SOURCE = "pride"  # or "massive"
DATASET_ID = "PXD012353"  # set dataset to explore
MAX_FILES = 5
MAKE_DENSE = False  # True can be large; use with care
MAX_DENSE_ELEMENTS = 5_000_000  # safety limit

def load_voxel_npz(path: Path):
    npz = np.load(path)
    coords = npz["coords"]
    vals = npz["vals"]
    if coords.size == 0:
        shape = (0, 0, 0)
    else:
        shape = tuple((coords.max(axis=0) + 1).tolist())
    return coords, vals, shape

def sparse_to_dense(coords, vals, shape):
    dense = np.zeros(shape, dtype=vals.dtype)
    for (pbin, fbin, tbin), v in zip(coords, vals):
        dense[pbin, fbin, tbin] = v
    return dense

voxel_dir = DATA_VOXEL / SOURCE / DATASET_ID
if not voxel_dir.exists():
    print("Voxel dir not found:", voxel_dir)
else:
    voxel_files = sorted(voxel_dir.glob("*.npz"))
    tensors = []
    for p in voxel_files[:MAX_FILES]:
        coords, vals, shape = load_voxel_npz(p)
        if coords.size == 0:
            dim_min = (0, 0, 0)
            dim_max = (0, 0, 0)
        else:
            dim_min = tuple(coords.min(axis=0).tolist())
            dim_max = tuple(coords.max(axis=0).tolist())
        item = {
            "file": p.name,
            "coords": coords,
            "vals": vals,
            "shape": shape,
            "dim_min": dim_min,
            "dim_max": dim_max,
        }
        if MAKE_DENSE and np.prod(shape) <= MAX_DENSE_ELEMENTS:
            item["dense"] = sparse_to_dense(coords, vals, shape)
        tensors.append(item)
    print("Loaded files:", len(tensors))
    if tensors:
        print("Example:", tensors[0]["file"], "shape=", tensors[0]["shape"], "nnz=", int(tensors[0]["vals"].size))
        print("Example dim_min:", tensors[0]["dim_min"], "dim_max:", tensors[0]["dim_max"])

    if tensors:
        all_mins = np.array([t["dim_min"] for t in tensors])
        all_maxs = np.array([t["dim_max"] for t in tensors])
        global_min = tuple(all_mins.min(axis=0).tolist())
        global_max = tuple(all_maxs.max(axis=0).tolist())
        print("Global dim_min:", global_min, "dim_max:", global_max)

Loaded files: 5
Example: 20140103_Velos1_HRV_MV_DCmin1.npz shape= (1445, 1900, 8099) nnz= 6987607


## Voxel-Only DIA/DDA Classification
Classify datasets by scanning voxel file names for DIA/DDA tokens. UNKNOWN if no clues.

In [ ]:
def classify_from_voxel_names(voxel_base: Path, max_files: int = 500):
    rows = []
    if not voxel_base.exists():
        print("Voxel base not found:", voxel_base)
        return rows
    for source_dir in sorted(voxel_base.iterdir()):
        if not source_dir.is_dir():
            continue
        source = source_dir.name
        for dataset_dir in sorted(source_dir.iterdir()):
            if not dataset_dir.is_dir():
                continue
            dataset = dataset_dir.name
            tokens = Counter()
            files = 0
            for p in dataset_dir.glob("*.npz"):
                files += 1
                name = p.name.upper()
                if "DIA" in name:
                    tokens["DIA"] += 1
                if "DDA" in name:
                    tokens["DDA"] += 1
                if files >= max_files:
                    break
            if tokens["DIA"] and tokens["DDA"]:
                label = "MIXED"
            elif tokens["DIA"]:
                label = "DIA"
            elif tokens["DDA"]:
                label = "DDA"
            else:
                label = "UNKNOWN"
            rows.append({
                "source": source,
                "dataset": dataset,
                "voxel_files": files,
                "dia_hits": tokens["DIA"],
                "dda_hits": tokens["DDA"],
                "label": label,
            })
    return rows

rows = classify_from_voxel_names(DATA_VOXEL, max_files=1000)
if rows:
    if pd is not None:
        df_class = pd.DataFrame(rows)
        display(df_class.sort_values(["label", "source", "dataset"]))
        display(df_class["label"].value_counts().to_frame("dataset_count"))
    else:
        label_counts = Counter(r["label"] for r in rows)
        print("Label counts:", dict(label_counts))
        for r in rows[:20]:
            print(r)

Label counts: {'UNKNOWN': 2}
{'source': 'pride', 'dataset': 'PXD010595', 'voxel_files': 0, 'dia_hits': 0, 'dda_hits': 0, 'label': 'UNKNOWN'}
{'source': 'pride', 'dataset': 'PXD012353', 'voxel_files': 23, 'dia_hits': 0, 'dda_hits': 0, 'label': 'UNKNOWN'}


## 8. Save Cleaned Snapshot for Next Notebook
Persist a cleaned subset for later steps.

In [ ]:
if df is None:
    print("Load a dataset first.")
else:
    cleaned = df.copy()
    # Example cleaning: drop fully empty columns
    cleaned = cleaned.dropna(axis=1, how="all")
    out_path = BASE_PATH / "data" / "exploration_cleaned_snapshot.csv"
    cleaned.to_csv(out_path, index=False)
    print("Saved:", out_path)

Load a dataset first.
